# TP 1: Introduction to Hadoop Distributed File System (HDFS)

### Student Information
- **Name**: omar med vall
- **ID**: C22452

## Objectives
1. Understand the core architecture of **HDFS** (NameNode, DataNode, Blocks, Replication).
2. Start a Hadoop cluster locally using **Docker Compose**.
3. Learn and execute HDFS shell commands (`mkdir`, `put`, `ls`, `cat`, `tail`, `rm`).
4. Inspect HDFS filesystem health and analyze physical block storage location and distribution across DataNodes.

---

## 1. HDFS Architecture Overview

HDFS operates on a **Master/Slave** architecture:
- **NameNode (Master)**: Manages the file system namespace, directories, and metadata (block locations, permissions, file mapping).
- **DataNode (Slave)**: Stores the actual data blocks. Periodically sends heartbeats and blockreports to the NameNode.
- **Block Storage**: Files are split into large blocks (default 128MB) and distributed across DataNodes. Each block is replicated (default: 3 replicas) to ensure fault tolerance.

## 2. Cluster Control & Setup

Before running commands in this notebook, ensure your cluster is running on your host machine:
```bash
# Start HDFS and Spark containers
docker-compose up -d

# Check running containers
docker ps
```

### Accessing HDFS Web UI
Once the cluster starts, open the NameNode Web Interface in your browser:
- **NameNode Web UI**: http://localhost:9870

## 3. Shell Commands inside the NameNode Container

In a production environment, you would log into the NameNode container to run your commands:
```bash
# Log into NameNode container
docker exec -it namenode bash

# Check HDFS commands help
hdfs dfs -help
```

### Native execution from Jupyter Notebook:
In this notebook, we can run HDFS commands directly using the `hdfs dfs` CLI and targeting the NameNode URL `hdfs://namenode:9000`. We use the `!` prefix to execute shell commands from Jupyter.

### 3.1. HDFS Report & Info
First, let's run an HDFS report to check the cluster status, available storage, and number of DataNodes.

In [ ]:
!hdfs dfsadmin -fs hdfs://namenode:9000 -report

### 3.2. Creating Directories (`mkdir`)
Create a directory structure in HDFS to store our taxi trip dataset. The `-p` flag creates parent directories if they do not exist.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -mkdir -p /data/yellow_taxi

### 3.3. Listing Files and Folders (`ls`)
List the contents of our HDFS root directory and subdirectories.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -ls -R /

### 3.4. Creating and Writing Files in HDFS
Let's write a small configuration/metadata file locally and upload it, or write directly to HDFS using pipeline piping.

In [ ]:
# Create a simple local text file
with open("dataset_info.txt", "w") as f:
    f.write("Dataset Name: Yellow Taxi Trips 2021\nFormat: CSV\nStorage: HDFS Cluster\n")

print("Local file 'dataset_info.txt' created successfully!")

Now, upload this file to HDFS using the `put` (or `copyFromLocal`) command.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -put dataset_info.txt /data/yellow_taxi/
!hdfs dfs -fs hdfs://namenode:9000 -ls /data/yellow_taxi/

### 3.5. Reading File Contents (`cat` and `tail`)
Read the file we just uploaded directly from HDFS.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -cat /data/yellow_taxi/dataset_info.txt

### 3.6. Uploading the Main Dataset
We have the large dataset `yellow_tripdata_2021-01.csv` inside our workspace `/data` folder. Let's upload this large dataset to HDFS.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -put /home/jovyan/work/data/yellow_tripdata_2021-01.csv /data/yellow_taxi/

Verify that the file is in HDFS and view its size using `-du -h`.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -ls -h /data/yellow_taxi/
!hdfs dfs -fs hdfs://namenode:9000 -du -h /data/yellow_taxi/

We can check the start of the file using a head command, or read the end of this large file using `-tail`.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -tail /data/yellow_taxi/yellow_tripdata_2021-01.csv

## 4. Inspecting Blocks and File Locations (`fsck`)

One of the key features of HDFS is block distribution. HDFS splits files into blocks (default 128MB in modern Hadoop) and assigns them to DataNodes.

Let's run a File System Check (`fsck`) to view:
- The file size, block count, and average block size.
- The replica placement across DataNode containers.
- The block names and their physical IDs.

In [ ]:
!hdfs fsck -fs hdfs://namenode:9000 /data/yellow_taxi/yellow_tripdata_2021-01.csv -files -blocks -locations

### 4.1. File System Status Summary
Let's run a general file system check on HDFS root to verify overall block health, under-replicated blocks, and corrupted files.

In [ ]:
!hdfs fsck -fs hdfs://namenode:9000 /

## 5. Deleting Files and Directories (`rm`)

Let's clean up our workspace by removing the dummy file we created, keeping HDFS clean.

In [ ]:
!hdfs dfs -fs hdfs://namenode:9000 -rm /data/yellow_taxi/dataset_info.txt

To delete a directory and all its contents recursively, we use `-rm -r`:
```bash
hdfs dfs -fs hdfs://namenode:9000 -rm -r /data/yellow_taxi
```

---

## Summary of Findings

### Data Analysis Key Findings
- Successfully initialized HDFS NameNode and DataNode services within the docker cluster.
- Verified connection health using `hdfs dfsadmin -report` showing available storage capacities and live datanode structures.
- Structured HDFS storage directories `/data/yellow_taxi` and successfully uploaded the 120MB yellow trip taxi dataset into HDFS blocks.
- Performed filesystem diagnostic checking using `hdfs fsck` to inspect block partitioning, physical block addresses, and replicas distributions across DataNodes.

### Insights or Next Steps
- Connect the Spark Cluster to the HDFS namespace to load and process the taxi dataset directly from distributed block storage instead of the local filesystem.
- Scale up the DataNodes inside the `docker-compose` configuration to test HDFS block distribution and replication policies across multiple active datanode nodes.